# Modellvergleich mit Kreuzvalidierung (RandomForest vs XGBoost vs CatBoost)

**Warum Kreuzvalidierung statt einem einzelnen Train/Test-Split?**

Der Datensatz hat nur ~1000 Zeilen. Bei einem einzelnen Split landen in der
Testmenge nur 200 Zeilen — und in Untergruppen (z. B. "leichte" Fälle wie
Diebstahl/geparktes Auto) teilweise nur 30-40 Zeilen. Bei so kleinen
Stichproben kann eine einzelne ungewöhnliche Zeile das R² stark verändern,
unabhängig davon, wie gut das Modell wirklich ist.

Kreuzvalidierung (K-Fold) teilt die Daten in *K* Teile (Folds). Das Modell
wird *K* Mal trainiert — jedes Mal auf *K-1* Teilen, getestet auf dem
verbleibenden Teil. Am Ende wird über alle *K* Ergebnisse gemittelt. So
landet jede Zeile irgendwann in der Testmenge, und die Bewertung hängt
nicht von einem einzigen "glücklichen" oder "unglücklichen" Split ab.

In diesem Notebook werden keine Metriken mehr nach Untergruppen
(leicht/Kollision) aufgeteilt — bei ~1000 Zeilen ist das pro Gruppe
weiterhin zu wenig, um verlässlich zu sein.

In [4]:
import sys
import os
from pathlib import Path

project_root = Path.cwd()
while not (project_root / "src").exists() and project_root != project_root.parent:
    project_root = project_root.parent

sys.path.insert(0, str(project_root))
os.chdir(project_root)

print("Project root:", project_root)
print("CWD now:", os.getcwd())

Project root: d:\visual-code-projects\InterGeeks-Agiles-Programmierprojekt
CWD now: d:\visual-code-projects\InterGeeks-Agiles-Programmierprojekt


In [5]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, cross_validate, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import make_scorer, mean_absolute_error, r2_score
import xgboost as xgb
from catboost import CatBoostRegressor

RANDOM_STATE = 42
N_SPLITS = 5  # 5-fache Kreuzvalidierung

TARGET = "vehicle_claim"
TARGET_FIELDS = [
    "auto_year",
    "auto_make",
    "incident_type",
    "incident_severity",
    "collision_type",
    "number_of_vehicles_involved",
    "witnesses",
    "police_report_available",
    "bodily_injuries",
    "property_damage",
]

# "?" bei collision_type bedeutet "nicht zutreffend" (Diebstahl/geparkt),
# kein echtes Rauschen -> explizit kodieren statt mit dem Modus füllen.
COLLISION_TYPE_NA_LABEL = "NotApplicable"

## 1. Laden & Bereinigen

**Wichtig:** Stellen Sie sicher, dass hier die echte, originale
`dataset.csv` mit ~1000 Zeilen geladen wird — nicht eine synthetisch
vergrößerte Version.

In [ ]:
def load_and_clean(path: str = "data/raw/dataset.csv"):
    raw_df = pd.read_csv(path)
    df = raw_df.copy()

    missing_markers = ["NA", "N/A", "null", "None", ""]
    df = df.replace(missing_markers, np.nan)

    if "collision_type" in df.columns:
        df["collision_type"] = df["collision_type"].replace("?", COLLISION_TYPE_NA_LABEL)

    df = df.replace("?", np.nan)
    df = df[TARGET_FIELDS + [TARGET]].copy()

    num_cols = df.select_dtypes(include=["number"]).columns.tolist()
    cat_cols = df.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    if TARGET in num_cols:
        num_cols.remove(TARGET)

    for col in num_cols:
        if df[col].isna().sum() > 0:
            df[col] = df[col].fillna(df[col].median())

    for col in cat_cols:
        if df[col].isna().sum() > 0:
            mode = df[col].mode(dropna=True)
            df[col] = df[col].fillna(mode.iloc[0] if len(mode) else "Unknown")

    return df, cat_cols


df, cat_cols = load_and_clean("data/raw/dataset.csv")
cat_cols = [c for c in cat_cols if c != TARGET]

print(f"Zeilen: {df.shape[0]}, Spalten: {df.shape[1]}")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'dataset.csv'

Kontrolle: Sollte ~1000 Zeilen zeigen. Falls hier eine viel größere Zahl
steht, wird vermutlich der falsche (synthetische) Datensatz geladen.

In [ ]:
assert df.shape[0] < 2000, (
    f"Datensatz hat {df.shape[0]} Zeilen — das sieht nach dem synthetischen, "
    "nicht dem originalen Datensatz aus. Bitte den Pfad prüfen."
)

## 2. Encodings vorbereiten

- **One-Hot** (für RandomForest / XGBoost)
- **Native kategorial** (für CatBoost, via `cat_features`)

Bei Kreuzvalidierung wird kein fester Train/Test-Split mehr gebraucht —
`cross_validate` übernimmt das Aufteilen in jedem Fold selbst.

In [ ]:
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)
X_encoded = df_encoded.drop(columns=[TARGET])
y_encoded = df_encoded[TARGET]

X_cat = df.drop(columns=[TARGET])
y_cat = df[TARGET]

print("One-Hot-Form:", X_encoded.shape)
print("Kategorial-Form:", X_cat.shape)

## 3. Scorer & Hilfsfunktion für die Ausgabe

In [ ]:
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "r2": "r2",
    "mae": make_scorer(mean_absolute_error, greater_is_better=False),
}


def print_cv_results(model_name: str, cv_results: dict):
    r2_scores = cv_results["test_r2"]
    mae_scores = -cv_results["test_mae"]  # war negativ wegen greater_is_better=False

    print(f"--- {model_name} ({N_SPLITS}-fache Kreuzvalidierung) ---")
    print(f"R²  pro Fold:  {np.round(r2_scores, 3)}")
    print(f"R²  Mittelwert: {r2_scores.mean():.4f}   (Std: {r2_scores.std():.4f})")
    print(f"MAE pro Fold:  {np.round(mae_scores, 0)}")
    print(f"MAE Mittelwert: {mae_scores.mean():.2f}   (Std: {mae_scores.std():.2f})")
    print()

## 4. Random Forest (Baseline)

In [ ]:
rf_model = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=RANDOM_STATE)
rf_cv = cross_validate(rf_model, X_encoded, y_encoded, cv=kf, scoring=scoring, n_jobs=-1)
print_cv_results("Random Forest", rf_cv)

## 5. XGBoost (Standardparameter, zum Vergleich)

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=100, max_depth=6, learning_rate=0.1, random_state=RANDOM_STATE
)
xgb_cv = cross_validate(xgb_model, X_encoded, y_encoded, cv=kf, scoring=scoring, n_jobs=-1)
print_cv_results("XGBoost (Standard)", xgb_cv)

## 6. CatBoost (native kategoriale Behandlung)

CatBoost lässt sich nicht direkt in `cross_validate` mit `cat_features`
einsetzen (sklearn kennt diesen Parameter nicht). Deshalb wird die
Kreuzvalidierung hier manuell mit derselben `KFold`-Aufteilung
durchgeführt, damit der Vergleich fair bleibt.

In [ ]:
cat_r2_scores = []
cat_mae_scores = []

for fold_i, (train_idx, test_idx) in enumerate(kf.split(X_cat), start=1):
    X_tr, X_te = X_cat.iloc[train_idx], X_cat.iloc[test_idx]
    y_tr, y_te = y_cat.iloc[train_idx], y_cat.iloc[test_idx]

    cat_model = CatBoostRegressor(
        iterations=100, learning_rate=0.1, depth=6, random_state=RANDOM_STATE, verbose=0
    )
    cat_model.fit(X_tr, y_tr, cat_features=cat_cols)
    preds = cat_model.predict(X_te)

    cat_r2_scores.append(r2_score(y_te, preds))
    cat_mae_scores.append(mean_absolute_error(y_te, preds))

cat_r2_scores = np.array(cat_r2_scores)
cat_mae_scores = np.array(cat_mae_scores)

print("--- CatBoost (native kategorial, manuelle 5-fache CV) ---")
print(f"R²  pro Fold:  {np.round(cat_r2_scores, 3)}")
print(f"R²  Mittelwert: {cat_r2_scores.mean():.4f}   (Std: {cat_r2_scores.std():.4f})")
print(f"MAE pro Fold:  {np.round(cat_mae_scores, 0)}")
print(f"MAE Mittelwert: {cat_mae_scores.mean():.2f}   (Std: {cat_mae_scores.std():.2f})")

## 7. XGBoost Hyperparameter-Suche

`RandomizedSearchCV` nutzt intern bereits Kreuzvalidierung (`cv=5`), das
bleibt unverändert. Bei ~1000 Zeilen ist es sinnvoll, `n_iter` nicht zu
hoch zu wählen, um eine Überanpassung an die konkrete CV-Aufteilung zu
vermeiden.

In [ ]:
param_distributions = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [1, 3, 5],
    "reg_lambda": [0.5, 1.0, 2.0, 5.0],
    "reg_alpha": [0, 0.1, 0.5, 1.0],
}

search = RandomizedSearchCV(
    estimator=xgb.XGBRegressor(random_state=RANDOM_STATE),
    param_distributions=param_distributions,
    n_iter=40,
    scoring="r2",
    cv=kf,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)
search.fit(X_encoded, y_encoded)

print(f"Bestes CV R²: {search.best_score_:.4f}")
print("Beste Parameter:")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")

## 8. Getuntes XGBoost-Modell — Kreuzvalidierung mit den besten Parametern

In [ ]:
best_xgb = xgb.XGBRegressor(**search.best_params_, random_state=RANDOM_STATE)
best_xgb_cv = cross_validate(best_xgb, X_encoded, y_encoded, cv=kf, scoring=scoring, n_jobs=-1)
print_cv_results("XGBoost (getunt)", best_xgb_cv)

## Fazit

Vergleichen Sie die **Mittelwerte** (nicht einzelne Folds) der vier
Modelle oben. Achten Sie auch auf die **Standardabweichung** zwischen den
Folds — eine hohe Std bedeutet, dass die Schätzung bei diesem
Datensatzumfang noch recht unsicher ist, unabhängig vom Modell.

Bei nur ~1000 Zeilen sind Unterschiede von 0.02-0.05 in R² zwischen den
Modellen oft nicht bedeutsam — schauen Sie auf die Größenordnung des
Unterschieds im Verhältnis zur Std der Folds.